# Chapter 5 — Pretraining on unlabeled data

Chapter 4 assembled a complete GPT architecture. This chapter measures its initial behavior, prepares language-model training data,
and develops the training loop needed to learn next-token prediction from unlabeled text.

## 5.1 Preparing an untrained GPT baseline

Instantiate the Chapter 4 `GPTModel` before changing any weights. GPT-2 small normally supports 1,024 tokens, but this chapter uses a
shorter `context_length=256` to reduce computation during learning experiments. The other architecture dimensions remain those of
GPT-2 small.

Calling `model.eval()` disables dropout for the baseline generation. It does not train the model or make its random parameters
meaningful; it only makes repeated inference deterministic.

In [1]:
import torch

from build_llms_from_scratch_companion.model import GPTConfig, GPTModel

# Use GPT-2 small dimensions with a shorter context length for this chapter.
GPT_CONFIG_124M = GPTConfig(
    vocab_size=50257,
    context_length=256,
    emb_dim=768,
    num_heads=12,
    num_layers=12,
    dropout_rate=0.1,
    qkv_bias=False,
)

# Reproduce the random initialization used for the pretraining baseline.
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
# Disable all dropout while evaluating text generation before training.
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

## 5.2 Converting between text and batched token IDs

The generation helper operates on token-ID tensors, while examples begin and end as text. Two small conversion helpers establish the
boundary:

```text
text -> tokenizer.encode -> (num_tokens,) -> unsqueeze(0)
     -> (batch_size=1, num_tokens)

(batch_size=1, num_tokens) -> squeeze(0) -> (num_tokens,)
     -> tokenizer.decode -> text
```

Allowing `<|endoftext|>` explicitly lets later chapter examples encode document boundaries with the GPT-2 tokenizer.

In [ ]:
import tiktoken

from build_llms_from_scratch_companion.generation import generate_text_simple


def text_to_token_ids(
    text: str,
    tokenizer: tiktoken.Encoding,
) -> torch.Tensor:
    """Encode one text string as a batch containing one token-ID sequence.

    Args:
        text: Text to encode into GPT-2 token IDs.
        tokenizer: Tokenizer that maps text to token IDs.

    Returns:
        Token IDs shaped `(batch_size=1, num_tokens)`.
    """
    # encoded contains num_tokens integer token IDs.
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    # (num_tokens,) -> (batch_size=1, num_tokens)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor


def token_ids_to_text(
    token_ids: torch.Tensor,
    tokenizer: tiktoken.Encoding,
) -> str:
    """Decode a batch containing one token-ID sequence back into text.

    Args:
        token_ids: Token IDs shaped `(batch_size=1, num_tokens)`.
        tokenizer: Tokenizer that maps token IDs back to text.

    Returns:
        Decoded text for the single sequence in the batch.
    """
    # Remove the size-one batch dimension: (1, num_tokens) -> (num_tokens,).
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

### Generating text before pretraining

Generate ten token IDs from a four-token prompt to establish the model's initial behavior. The result has fourteen token positions,
and the original four prompt tokens remain at the beginning.

Because the model is randomly initialized, incoherent text is expected. Pretraining will adjust its parameters so the next-token
logits increasingly reflect patterns in the training text.

In [3]:
start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")
# input_ids: (batch_size=1, num_tokens=4)
input_ids = text_to_token_ids(start_context, tokenizer)
# token_ids: (batch_size=1, num_tokens=4 + max_new_tokens=10)
token_ids = generate_text_simple(
    model=model,
    idx=input_ids,
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M.context_length,
)
print("Output text:")
print(token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


## 5.3 Preparing inputs and targets for next-token prediction

Language-model pretraining uses the text itself as supervision. Each input token asks the model to predict the token immediately
after it. Begin with two short sequences arranged as a `(batch_size=2, num_tokens=3)` tensor.

In [5]:
# inputs: (batch_size=2, num_tokens=3)
# Each row contains one three-token input sequence.
inputs = torch.tensor(
    [
        [16833, 3626, 6100],  # "every effort moves"
        [40, 1107, 588],      # "I really like"
    ]
)

### Shifting inputs to create targets

Targets contain the same text shifted forward by one position:

```text
input:   every  effort  moves
target:         effort  moves  you
```

Input and target tensors have identical `(batch_size, num_tokens)` shapes, but each aligned pair represents one next-token training
example. Two rows of three positions therefore provide six predictions for the loss.

In [6]:
# targets: (batch_size=2, num_tokens=3)
# Each target row is its input row shifted one token forward.
targets = torch.tensor(
    [
        [3626, 6100, 345],    # " effort moves you"
        [1107, 588, 11311],   # " really like chocolate"
    ]
)

## 5.4 Converting logits into token probabilities

The model processes every input position in parallel and returns one unnormalized score for every vocabulary entry. Applying softmax
along the final `vocab_size` dimension turns each score vector into a probability distribution whose values sum to one:

```text
inputs   (batch_size, num_tokens)
logits   (batch_size, num_tokens, vocab_size)
probas   (batch_size, num_tokens, vocab_size)
```

In [7]:
# Evaluate the untrained model without constructing a gradient graph.
with torch.no_grad():
    # logits: (batch_size=2, num_tokens=3, vocab_size=50_257)
    logits = model(inputs)
# Normalize each position's vocabulary logits independently.
# probas: (batch_size=2, num_tokens=3, vocab_size=50_257)
probas = torch.softmax(logits, dim=-1)
probas.shape

torch.Size([2, 3, 50257])


### Inspecting greedy predictions at every position

Apply `argmax` across `vocab_size` to inspect the model's most probable token ID at each input position. Unlike generation, which uses
only the final position, loss calculation evaluates all positions in parallel.

Keeping the reduced dimension produces `(batch_size, num_tokens, 1)` rather than `(batch_size, num_tokens)`.

In [8]:
# Select the highest-probability vocabulary entry at every position.
# token_ids: (batch_size=2, num_tokens=3, 1)
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Token IDs:")
print(token_ids)

Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


### Comparing predicted and target text

Decode the first row's expected and predicted token IDs. The random model's predictions differ from the targets, which is expected
before pretraining. Flattening the prediction removes the trailing size-one dimension required only by the preceding `argmax`
example.

In [9]:
# targets[0]: (num_tokens=3,)
print(f"Targets batch 1: {token_ids_to_text(targets[0], tokenizer)}")
# token_ids[0]: (num_tokens=3, 1) -> flatten(): (num_tokens=3,)
print(
    "Outputs batch 1: "
    f"{token_ids_to_text(token_ids[0].flatten(), tokenizer)}"
)

Targets batch 1:  effort moves you
Outputs batch 1:  Armed heNetflix


## 5.5 Extracting the correct-token probabilities

Training does not optimize only the model's largest prediction. For every position, it selects the probability assigned to the known
`target_ids` value.

The three index components identify matching coordinates in `probas`:

```text
probas[text_idx, token positions, target token IDs]
       one row     [0, 1, 2]    one ID per position
```

This advanced indexing returns one correct-token probability per position, shaped `(num_tokens,)`. An untrained model assigns very
small probabilities to these targets because its probability mass is spread across `vocab_size=50_257` possibilities.

In [10]:
# Select the probability assigned to each correct target token in row 0.
text_idx = 0
# target_probas_1: (num_tokens=3,)
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1:", target_probas_1)

# Repeat the same position-by-position lookup for row 1.
text_idx = 1
# target_probas_2: (num_tokens=3,)
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2:", target_probas_2)

Text 1: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


## Chapter 5 summary

The chapter now establishes the quantities needed to measure an untrained language model:

- text is converted to batched token IDs and decoded through explicit shape transformations;
- each `targets` row is the corresponding `inputs` row shifted forward by one token;
- GPT predicts all next-token positions in parallel as `(batch_size, num_tokens, vocab_size)` logits;
- softmax converts vocabulary logits into per-position probability distributions; and
- advanced indexing extracts the probability assigned to the correct target token at every position.

These target probabilities lead directly to the logarithmic loss used for pretraining.